In [ ]:
# Run this cell first
!pip install praw psaw newsapi-python vaderSentiment transformers datasets sentence-transformers scikit-learn xgboost shap lime
# newspaper3k may need extra downloads (done later)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 85.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 10.1 MB/s eta 0:00:00
  Created wheel for tinysegmenter: filename=tinysegmenter-0.3-py3-none-any.whl size=13540 sha256=67afbf811e353efe8bb7fdc131cf0d4ec91c12dea5c89611b1454ca280b56787
  Stored in directory: /root/.cache/pip/wheels/a5/91/9f/00d66475960891a

In [ ]:
import os, re, json, time, math
import pandas as pd
import numpy as np
from datetime import datetime, timezone
import matplotlib.pyplot as plt
import joblib
from collections import Counter


# NLP
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# VADER
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# HuggingFace pipeline (for inference)
from transformers import pipeline

# sentence-transformers
from sentence_transformers import SentenceTransformer

import praw
from psaw import PushshiftAPI

from newsapi import NewsApiClient

from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
path = "/content/drive/MyDrive/capstone"

In [ ]:
from google.colab import userdata
REDDIT_CLIENT_ID = userdata.get('REDDIT_CLIENT_ID')
REDDIT_CLIENT_SECRET = userdata.get('REDDIT_CLIENT_SECRET')
REDDIT_USER_AGENT = userdata.get('REDDIT_USER_AGENT')
NEWS_API_KEY = userdata.get("NEWS_API_KEY")

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

In [ ]:
def clean_text(text):
    if not isinstance(text, str): return ""
    text = text.lower()
    text = re.sub(r'http\S+',' ', text)           # remove urls
    text = re.sub(r'@\w+',' ', text)              # twitter mentions
    text = re.sub(r'\$[A-Za-z]{1,6}\b',' ', text) # $TICKER
    text = re.sub(r'[^a-z0-9\s]',' ', text)       # keep alnum + spaces
    tokens = nltk.word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t)>1]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

In [ ]:
# crude ticker/company extractor: pass a list of tickers & company names you care about
def find_tickers(text, tickers_list, company_list=None):
    found = set()
    text_up = text.upper()
    # ticker forms: "TSLA", "$TSLA", "Tesla"
    for t in tickers_list:
        if re.search(r'\b' + re.escape(t.upper()) + r'\b', text_up):
            found.add(t.upper())
    if company_list:
        for name, t in company_list.items():
            if name.lower() in text.lower():
                found.add(t.upper())
    return list(found)

In [ ]:
reddit = praw.Reddit(client_id=REDDIT_CLIENT_ID,
                     client_secret=REDDIT_CLIENT_SECRET,
                     user_agent=REDDIT_USER_AGENT)

TECHNICAL SECTOR COMPANIES

In [ ]:
tickers = ["TSLA"]   # change to your portfolio
subs = []
start_epoch = int(datetime(2024,1,1).timestamp())
end_epoch   = int(datetime(2025,9,1).timestamp())

In [ ]:
def get_reddit_posts(keyword: str):
    """
    Fetch top Reddit posts from r/stocks, r/investing, r/wallstreetbets
    relevant to a keyword (last 1 year).
    """
    try:
        results = []
        subreddits = ["stocks", "investing", "wallstreetbets"]

        for sub in subreddits:
            subreddit = reddit.subreddit(sub)
            for post in subreddit.search(keyword, sort="relevance", time_filter="year", limit=50):
                text = (post.title or "") + " " + (post.selftext or "")
                results.append({
                    "id": post.id,
                    "created_utc": datetime.fromtimestamp(post.created_utc, tz=timezone.utc),
                    "subreddit": sub,
                    "score": post.score,
                    "num_comments": post.num_comments,
                    "title": post.title,
                    "body": post.selftext,
                    "url": f"https://www.reddit.com{post.permalink}",
                    "clean_text": clean_text(text)
                })

        return results if results else []

    except Exception as e:
        print(f"Error fetching Reddit posts: {str(e)}")
        return []


In [ ]:
all_subs = []
for t in tickers:
    posts = get_reddit_posts(t)
    if isinstance(posts, list):   # ensure it's not an error string
        all_subs.extend(posts)

# Convert to DataFrame
df_reddit = pd.DataFrame(all_subs)
print("reddit rows:", len(df_reddit))
print(df_reddit.head())

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



reddit rows: 150
        id               created_utc subreddit  score  num_comments  \
0  1nbo9zy 2025-09-08 13:50:34+00:00    stocks    745           161   
1  1jfn3wn 2025-03-20 11:53:18+00:00    stocks  14663           666   
2  1j9qzgg 2025-03-12 18:25:51+00:00    stocks   7231          1252   
3  1j7l0bv 2025-03-09 23:35:58+00:00    stocks  18878           490   
4  1jaciwt 2025-03-13 13:55:53+00:00    stocks   4267           494   

                                               title  \
0                  TSLA to lose major revenue source   
1  Tesla (TSLA) accounting raises red flags as re...   
2                             TSLA investors, beware   
3  TSLA being investigated by Transport Canada fo...   
4  Tesla (TSLA) Stock: Trump’s Purchase Fails to ...   

                                                body  \
0  Bloomberg is reporting that EV deregulation fr...   
1  “Tesla’s (TSLA) accounting practices are raisi...   
2  Trump's support of Tesla is a desperate and la..

In [ ]:
df_reddit

,id,created_utc,subreddit,score,num_comments,title,body,url,clean_text
0,1nbo9zy,2025-09-08 13:50:34+00:00,stocks,745,161,TSLA to lose major revenue source,Bloomberg is reporting that EV deregulation fr...,https://www.reddit.com/r/stocks/comments/1nbo9...,tsla lose major revenue source bloomberg repor...
1,1jfn3wn,2025-03-20 11:53:18+00:00,stocks,14663,666,Tesla (TSLA) accounting raises red flags as re...,“Tesla’s (TSLA) accounting practices are raisi...,https://www.reddit.com/r/stocks/comments/1jfn3...,tesla tsla accounting raise red flag report sh...
2,1j9qzgg,2025-03-12 18:25:51+00:00,stocks,7231,1252,"TSLA investors, beware",Trump's support of Tesla is a desperate and la...,https://www.reddit.com/r/stocks/comments/1j9qz...,tsla investor beware trump support tesla despe...
3,1j7l0bv,2025-03-09 23:35:58+00:00,stocks,18878,490,TSLA being investigated by Transport Canada fo...,The article notes that **four Tesla dealership...,https://www.reddit.com/r/stocks/comments/1j7l0...,tsla investigated transport canada cooking boo...
4,1jaciwt,2025-03-13 13:55:53+00:00,stocks,4267,494,Tesla (TSLA) Stock: Trump’s Purchase Fails to ...,Who knew that the publicity stunt on the WH la...,https://www.reddit.com/r/stocks/comments/1jaci...,tesla tsla stock trump purchase fails sustain ...
...,...,...,...,...,...,...,...,...,...
145,1hbzq72,2024-12-11 18:05:13+00:00,wallstreetbets,1208,99,TSLA 💎 Hands 950%,Held through ATH to 100s back to ATH,https://www.reddit.com/r/wallstreetbets/commen...,tsla hand 950 held ath 100 back ath
146,1fuhafq,2024-10-02 14:18:37+00:00,wallstreetbets,388,407,$98k YOLO on $TSLA 10/18 calls. Tesla is going...,Not financial advice,https://www.reddit.com/r/wallstreetbets/commen...,98k yolo 10 18 call tesla going print cybercab...
147,1gp6tf7,2024-11-11 23:38:58+00:00,wallstreetbets,957,123,$TSLA Calls Closed +$400k,Haven't gotten any sleep holding these Tesla c...,https://www.reddit.com/r/wallstreetbets/commen...,call closed 400k gotten sleep holding tesla ca...
148,1k5ltx6,2025-04-23 00:17:43+00:00,wallstreetbets,26572,1641,THIS CASINO IS RIGGED!,EARNING MISSED BY WHOPPING 35%? TSLA IS UP!!!!...,https://www.reddit.com/r/wallstreetbets/commen...,casino rigged earning missed whopping 35 tsla ...


In [ ]:
def get_market_news_df(query: str) -> pd.DataFrame:
    """
    Fetch the most relevant business/finance news about a company or sector.
    Returns a DataFrame with title, url, publishedAt, and source.
    """
    try:
        if not NEWS_API_KEY:
            return pd.DataFrame([{"error": "NEWS_API_KEY not set in environment"}])

        newsapi = NewsApiClient(api_key=NEWS_API_KEY)

        # Fetch relevant articles
        articles = newsapi.get_everything(
            q=query,
            language="en",
            sort_by="publishedAt"
        )

        if not articles["articles"]:
            return pd.DataFrame([{"error": f"No news found for {query}"}])

        # Build DataFrame
        df = pd.DataFrame([{
            "title": a["title"],
            "url": a["url"],
            "publishedAt": a["publishedAt"],
            "source": a["source"]["name"]
        } for a in articles["articles"][:10]])

        return df

    except Exception as e:
        return pd.DataFrame([{"error": str(e)}])

In [ ]:
df_news = get_market_news_df("Tesla")
print("rows:", len(df_news))
df_news.head()

rows: 10


,title,url,publishedAt,source
0,Top Large Cap Stocks To Follow Now – September...,https://www.etfdailynews.com/2025/09/12/top-la...,2025-09-12T12:30:46Z,ETF Daily News
1,Could Elon Musk Buy Verizon? He Says It’s Poss...,https://www.androidheadlines.com/2025/09/could...,2025-09-12T12:20:25Z,Android Headlines
2,CHTR SECURITIES NOTICE: Did Charter Communicat...,https://www.globenewswire.com/news-release/202...,2025-09-12T12:18:00Z,GlobeNewswire
3,"RXST SECURITIES NOTICE: Did RxSight, Inc. Misl...",https://www.globenewswire.com/news-release/202...,2025-09-12T12:18:00Z,GlobeNewswire
4,"SLP SECURITIES NOTICE: Did Simulations Plus, I...",https://www.globenewswire.com/news-release/202...,2025-09-12T12:18:00Z,GlobeNewswire


In [ ]:
# --- Reddit prep ---
if not df_reddit.empty:
    df_reddit['source'] = 'reddit'
    df_reddit['date'] = pd.to_datetime(df_reddit['created_utc'], unit='s').dt.date
    df_reddit['text'] = df_reddit['title'].fillna('') + " " + df_reddit.get('body', '').fillna('')
    df_reddit['score'] = df_reddit.get('score', 0)
    df_reddit['num_comments'] = df_reddit.get('num_comments', 0)
    df_reddit['clean_text'] = df_reddit['text'].apply(clean_text)
else:
    df_reddit = pd.DataFrame(columns=['source','date','text','clean_text','score','num_comments'])

# --- News prep ---
if not df_news.empty and "error" not in df_news.columns:
    df_news['source'] = 'news'
    df_news['date'] = pd.to_datetime(df_news['publishedAt']).dt.date
    df_news['text'] = df_news['title'].fillna('')
    df_news['score'] = None
    df_news['num_comments'] = None
    df_news['clean_text'] = df_news['text'].apply(clean_text)
else:
    df_news = pd.DataFrame(columns=['source','date','text','clean_text','score','num_comments'])

# --- Merge all ---
df_all = pd.concat([df_reddit[['source','date','text','clean_text','score','num_comments']],
                    df_news[['source','date','text','clean_text','score','num_comments']]],
                   ignore_index=True, sort=False)

# shuffle rows
df_all = df_all.sample(frac=1, random_state=42).reset_index(drop=True)

print("Merged dataset rows:", len(df_all))
df_all.head()



Merged dataset rows: 160


,source,date,text,clean_text,score,num_comments
0,reddit,2025-05-09,LFG!!!!! TSLA this morning GOT ME INTO 6 FIGUR...,lfg tsla morning got figure insane month tradi...,3243,857
1,reddit,2025-03-25,"Stop buying $TSLA puts Hello Regard,\n\nI have...",stop buying put hello regard consistently buyi...,4284,429
2,reddit,2025-01-18,I Paper Handed My Way Out of Scoring Nearly $2...,paper handed way scoring nearly 200k tsla put ...,730,197
3,reddit,2025-02-07,Chinese Markets are Rejecting Tesla China Is M...,chinese market rejecting tesla china moving te...,1797,312
4,reddit,2025-06-21,Stocks to hold 5-10 years I’m looking for stoc...,stock hold 10 year looking stock hold 10 year ...,0,89


In [ ]:
os.makedirs(path, exist_ok=True)

# File path
merged_file = os.path.join(path, "merged_reddit_news.parquet")

# Save
df_all.to_parquet(merged_file, index=False)
print(f"Merged dataset saved → {merged_file}")

Merged dataset saved → /content/drive/MyDrive/capstone/merged_reddit_news.parquet


In [ ]:
if os.path.exists(merged_file):
    df_all = pd.read_parquet(merged_file)
    print(f"Loaded {len(df_all)} rows from {merged_file}")
else:
    print("File not found:", merged_file)

Loaded 160 rows from /content/drive/MyDrive/capstone/merged_reddit_news.parquet


In [ ]:
df_all2 = df_all.copy()

In [ ]:
analyzer = SentimentIntensityAnalyzer()
df_all['vader_compound'] = df_all['clean_text'].apply(lambda t: analyzer.polarity_scores(t)['compound'])
# simple labels from vader
df_all['vader_label'] = df_all['vader_compound'].apply(lambda s: 'pos' if s>=0.05 else ('neg' if s<=-0.05 else 'neu'))

# Finance note: VADER is tuned for social media — for financial text consider FinBERT or Loughran-McDonald lexicon.
print(df_all['vader_label'].value_counts())
df_all.head()


vader_label
pos    98
neg    41
neu    21
Name: count, dtype: int64


,source,date,text,clean_text,score,num_comments,vader_compound,vader_label
0,reddit,2025-05-09,LFG!!!!! TSLA this morning GOT ME INTO 6 FIGUR...,lfg tsla morning got figure insane month tradi...,3243.0,857.0,-0.9118,neg
1,reddit,2025-03-25,"Stop buying $TSLA puts Hello Regard,\n\nI have...",stop buying put hello regard consistently buyi...,4284.0,429.0,-0.9578,neg
2,reddit,2025-01-18,I Paper Handed My Way Out of Scoring Nearly $2...,paper handed way scoring nearly 200k tsla put ...,730.0,197.0,-0.7645,neg
3,reddit,2025-02-07,Chinese Markets are Rejecting Tesla China Is M...,chinese market rejecting tesla china moving te...,1797.0,312.0,0.7569,pos
4,reddit,2025-06-21,Stocks to hold 5-10 years I’m looking for stoc...,stock hold 10 year looking stock hold 10 year ...,0.0,89.0,0.8126,pos


In [ ]:
# Baseline: use VADER labels as weak labels
df_train = df_all[df_all['vader_label'].isin(['pos','neg'])].copy()  # remove neutral for binary
X = df_train['clean_text']
y = df_train['vader_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF
vect = TfidfVectorizer(ngram_range=(1,2), max_features=20000)
X_train_vec = vect.fit_transform(X_train)
X_test_vec = vect.transform(X_test)

# Logistic Regression
clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train_vec, y_train)

# Evaluate
y_pred = clf.predict(X_test_vec)
print(classification_report(y_test, y_pred))

joblib.dump(clf, os.path.join(path, "sentiment_model.pkl"))
joblib.dump(vect, os.path.join(path, "tfidf_vectorizer.pkl"))

print(f"Model + vectorizer saved → {path}")

              precision    recall  f1-score   support

         neg       0.50      0.12      0.20         8
         pos       0.73      0.95      0.83        20

    accuracy                           0.71        28
   macro avg       0.62      0.54      0.51        28
weighted avg       0.66      0.71      0.65        28

Model + vectorizer saved → /content/drive/MyDrive/capstone


In [ ]:
# Load FinBERT (pretrained on financial text)
model_name = "yiyanghkust/finbert-tone"  # popular FinBERT model for finance sentiment
finbert_pipe = pipeline(
    "sentiment-analysis",
    model=model_name,
    tokenizer=model_name,
    device=0 if torch.cuda.is_available() else -1  # GPU if available
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/533 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cpu


In [ ]:
# Helper function to get label + score
def finbert_sentiment(text):
    # truncate long text
    out = finbert_pipe(text[:512])[0]
    # output format: {'label': 'positive', 'score': 0.99}
    return out['label'].lower(), out['score']


In [ ]:
df_all['finbert_label_score'] = df_all['clean_text'].apply(lambda t: finbert_sentiment(t))
df_all[['finbert_label','finbert_score']] = pd.DataFrame(df_all['finbert_label_score'].tolist(), index=df_all.index)
df_all.drop(columns=['finbert_label_score'], inplace=True)

# Check
df_all[['text','finbert_label','finbert_score']].head()

,text,finbert_label,finbert_score
0,LFG!!!!! TSLA this morning GOT ME INTO 6 FIGUR...,neutral,1.000000
1,"Stop buying $TSLA puts Hello Regard,\n\nI have...",neutral,0.737032
2,I Paper Handed My Way Out of Scoring Nearly $2...,neutral,0.908910
3,Chinese Markets are Rejecting Tesla China Is M...,positive,0.955875
4,Stocks to hold 5-10 years I’m looking for stoc...,positive,1.000000


In [ ]:
# Count FinBERT sentiment distribution
print(df_all['finbert_label'].value_counts())

# If you want it neatly formatted
print("\nFinBERT Sentiment Counts:")
for label, count in df_all['finbert_label'].value_counts().items():
    print(f"{label}    {count}")

finbert_label
neutral     89
positive    40
negative    31
Name: count, dtype: int64

FinBERT Sentiment Counts:
neutral    89
positive    40
negative    31


In [ ]:
def top_tfidf_ngrams(docs, top_k=20):
    vect = TfidfVectorizer(ngram_range=(1,2), stop_words='english', max_features=5000)
    X = vect.fit_transform(docs)
    means = np.array(X.mean(axis=0)).ravel()
    top = sorted(zip(vect.get_feature_names_out(), means), key=lambda x: x[1], reverse=True)[:top_k]
    return [w for w, _ in top]

# Split docs by FinBERT label
pos_docs = df_all[df_all['finbert_label']=='positive']['clean_text'].astype(str)
neg_docs = df_all[df_all['finbert_label']=='negative']['clean_text'].astype(str)
neu_docs = df_all[df_all['finbert_label']=='neutral']['clean_text'].astype(str)

# Run for each FinBERT label
pos_top = top_tfidf_ngrams(pos_docs, 20)
neg_top = top_tfidf_ngrams(neg_docs, 20)
neu_top = top_tfidf_ngrams(neu_docs, 20)

# Print as clean lists
print("\nTop Positive ngrams:", pos_top)
print("\nTop Negative ngrams:", neg_top)
print("\nTop Neutral ngrams:", neu_top)


Top Positive ngrams: ['tesla', 'tsla', 'stock', 'year', 'market', 'price', 'best', 'elon', 'china', 'musk', 'buy', 'company', 'ai', '10', 'growth', 'long', 'going', 'gain', '10 year', 'think']

Top Negative ngrams: ['tesla', 'tsla', 'stock', 'investment', 'bfa', 'bfa law', 'contact', 'contact bfa', 'investor contact', 'law', 'law lost', 'mislead', 'mislead investor', 'money investment', 'security', 'security notice', 'lost', 'musk', 'investor', 'money']

Top Neutral ngrams: ['tsla', 'tesla', 'stock', 'year', 'elon', 'musk', 'company', 'buy', 'market', 'trump', '000', 'going', 'money', 'time', 'like', 'today', 'bought', '10', 'share', 'day']


In [ ]:
def top_features_by_class(vectorizer, clf, top_n=20):
    feature_names = np.array(vectorizer.get_feature_names_out())
    coefs = clf.coef_[0]

    # Top positive n-grams
    top_pos_idx = np.argsort(coefs)[-top_n:]
    top_pos = feature_names[top_pos_idx][::-1]

    # Top negative n-grams
    top_neg_idx = np.argsort(coefs)[:top_n]
    top_neg = feature_names[top_neg_idx]

    return list(top_pos), list(top_neg)

top_pos, top_neg = top_features_by_class(vect, clf, top_n=20)

print("Top Positive ngrams:", top_pos)
print("Top Negative ngrams:", top_neg)

Top Positive ngrams: ['year', '000', 'elon musk', 'company', 'term', 'yolo', 'growth', 'share', 'best', 'amazon', 'top', 'close', 'long', 'profit', '10', 'bear', 'yield', 'brkb', 'rivian', 'meta']
Top Negative ngrams: ['put', 'regret', 'canada', 'stupid bought', 'bought tsla', 'trump', 'early', 'stupid', 'tsla', 'bought', '250', 'selling', 'rebate', 'earning', 'regret selling', 'tsmc', 'ceo', 'reddit', '00', '400 call']


In [ ]:
# Simple tokenizer
def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

# Function to get top n tokens per sentiment
def get_top_ngrams_by_sentiment(df, label, n=20):
    texts = df[df['vader_label'] == label]['clean_text']
    tokens = []
    for txt in texts:
        tokens.extend(tokenize(txt))
    counter = Counter(tokens)
    return [w for w, _ in counter.most_common(n)]

# Print top ngrams
print("Top Positive ngrams:", get_top_ngrams_by_sentiment(df_all, "pos"))
print("Top Negative ngrams:", get_top_ngrams_by_sentiment(df_all, "neg"))
print("Top Neutral ngrams:", get_top_ngrams_by_sentiment(df_all, "neu"))

Top Positive ngrams: ['tesla', 'year', 'tsla', 'stock', 'market', 'company', '10', 'like', 'musk', 'time', 'elon', 'share', 'long', 'buy', 'price', 'would', 'going', 'term', 'bond', 'yield']
Top Negative ngrams: ['tsla', 'tesla', 'stock', 'trump', 'put', 'tariff', 'u', 'elon', 'said', 'musk', 'canada', 'one', 'would', 'market', 'trade', 'time', 'going', 'china', 'even', 'still']
Top Neutral ngrams: ['money', 'lost', 'tsla', 'investment', 'security', 'notice', 'mislead', 'investor', 'contact', 'bfa', 'law', 'today', 'year', 'inc', 'vehicle', 'musk', 'tesla', 'delivery', 'put', 'stock']


In [ ]:
# 1. Logistic Regression (already trained)
y_pred_lr = clf.predict(X_test_vec)
print("Logistic Regression Report:")
print(classification_report(y_test, y_pred_lr))


Logistic Regression Report:
              precision    recall  f1-score   support

         neg       0.50      0.12      0.20         8
         pos       0.73      0.95      0.83        20

    accuracy                           0.71        28
   macro avg       0.62      0.54      0.51        28
weighted avg       0.66      0.71      0.65        28



# Model comparison and reasoning — VADER vs LogisticRegression vs FinBERT

**Summary of what we did**
- We scored the same TSLA dataset with three approaches:
  1. **VADER** — a rule/lexicon-based social-media tuned scorer (fast, no training).
  2. **Logistic Regression + TF-IDF** — trained on Financial PhraseBank (phrasebank) as a lightweight, explainable ML model.
  3. **FinBERT** — a finance-domain transformer (strong out-of-the-box financial understanding).

**Observed behavior**  
- **Logistic Regression**: performs *well* on formal, news-like sentences. It is fast, interpretable, and easy to export & run. It **learns the patterns from labeled financial text**, but will underperform on very informal social-media language unless we augment training data.
- **VADER**: quick baseline for social text, but **too noisy for finance** and often confuses domain-specific terms (e.g., “dilution”, “guidance”) or sarcasm. Good for sanity checks, not final signals.
- **FinBERT**: strong, domain-tuned transformer that captures finance nuance better than VADER or a TF-IDF model trained on limited data. Downsides: heavier (latency, GPU/CPU cost), less interpretable, and in some settings there may be constraints around directly using such large transformer models.

Now we use **Logistic Regression (trained on Financial PhraseBank)** as the primary model for **news/article** sentiment: it’s fast, explainable, and easy to ship in a microservice or agent.  

**TRAINING LOGISTIC REGRESSION ON PHRASE BANK**



In [ ]:
df = pd.read_csv("/content/drive/MyDrive/capstone/financialData.csv", encoding='latin-1')

In [ ]:
# Rename columns to 'text' and 'sentiment' based on the file content
df = df.rename(columns={
    'neutral': 'sentiment',
    'According to Gran , the company has no plans to move all production to Russia , although that is where the company is growing .': 'text'
})

df = df[['text', 'sentiment']]  # keep only relevant columns
df.head()

,text,sentiment
0,Technopolis plans to develop in stages an area...,neutral
1,The international electronic industry company ...,negative
2,With the new production plant the company woul...,positive
3,According to the company 's updated strategy f...,positive
4,FINANCING OF ASPOCOMP 'S GROWTH Aspocomp is ag...,positive


In [ ]:
df2 = df.copy()

In [ ]:
df_train = df[df['sentiment'].isin(['positive','negative'])].copy()

X = df_train['text']
y = df_train['sentiment'].str.lower()  # convert to 'pos'/'neg'

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# TF-IDF vectorization
vect = TfidfVectorizer(ngram_range=(1,2), max_features=20000)
X_train_vec = vect.fit_transform(X_train)
X_test_vec = vect.transform(X_test)

# Logistic Regression
clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train_vec, y_train)

# Evaluate
y_pred = clf.predict(X_test_vec)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    negative       0.77      0.76      0.77       121
    positive       0.89      0.90      0.90       273

    accuracy                           0.86       394
   macro avg       0.83      0.83      0.83       394
weighted avg       0.86      0.86      0.86       394



In [ ]:
joblib.dump(clf, "/content/logreg_sentiment.pkl")
joblib.dump(vect, "/content/tfidf_vectorizer2.pkl")
print("Model and vectorizer saved!")

Model and vectorizer saved!


In [ ]:
article_text = "TSLA to lose major revenue source"

# Transform text into TF-IDF features
X_new_vec = vect.transform([article_text])

# Predict sentiment
predicted_sentiment = clf.predict(X_new_vec)[0]
print(f"Predicted sentiment: {predicted_sentiment}")

Predicted sentiment: positive


In [ ]:
article_text = "TSLA being investigated by Transport Canada for cooking their books in Canada to snag EV rebates without selling cars"

# Transform text into TF-IDF features
X_new_vec = vect.transform([article_text])

# Predict sentiment
predicted_sentiment = clf.predict(X_new_vec)[0]
print(f"Predicted sentiment: {predicted_sentiment}")

Predicted sentiment: positive


In [ ]:
article_text = "Tesla (TSLA) accounting raises red flags as report shows $1.4 billion missing"

# Transform text into TF-IDF features
X_new_vec = vect.transform([article_text])

# Predict sentiment
predicted_sentiment = clf.predict(X_new_vec)[0]
print(f"Predicted sentiment: {predicted_sentiment}")

Predicted sentiment: negative


In [ ]:
sentence = "TSLA to lose major revenue source"

# Use your FinBERT helper
label, score = finbert_sentiment(sentence)

print(f"Sentence: {sentence}")
print(f"Predicted Sentiment: {label}, Confidence: {score:.2f}")


Sentence: TSLA being investigated by Transport Canada for cooking their books in Canada to snag EV rebates without selling cars.
Predicted Sentiment: neutral, Confidence: 1.00


In [ ]:
sentence = "TSLA being investigated by Transport Canada for cooking their books in Canada to snag EV rebates without selling cars"

# Use your FinBERT helper
label, score = finbert_sentiment(sentence)

print(f"Sentence: {sentence}")
print(f"Predicted Sentiment: {label}, Confidence: {score:.2f}")


Sentence: TSLA being investigated by Transport Canada for cooking their books in Canada to snag EV rebates without selling cars
Predicted Sentiment: neutral, Confidence: 0.99


In [ ]:
sentence = "Tesla (TSLA) accounting raises red flags as report shows $1.4 billion missing"

# Use your FinBERT helper
label, score = finbert_sentiment(sentence)

print(f"Sentence: {sentence}")
print(f"Predicted Sentiment: {label}, Confidence: {score:.2f}")

Sentence: Tesla (TSLA) accounting raises red flags as report shows $1.4 billion missing
Predicted Sentiment: positive, Confidence: 0.46


In [ ]:
df_reddit['title'][1]

'Tesla (TSLA) accounting raises red flags as report shows $1.4 billion missing'